In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 60.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=fbdb412ace28d77f83b0bfe2a47b87020c578a7c964a8a12ac6865b9f0820c77
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [4]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# --- Helper Function ---
def get_quantum_random_bits(num_bits):
    """Generates a list of random bits by measuring a quantum superposition."""
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits))
    qc.measure(range(num_bits), range(num_bits))

    backend = BasicSimulator()
    t_qc = transpile(qc, backend)
    result = backend.run(t_qc, shots=1).result()
    counts = result.get_counts()
    bitstring = list(counts.keys())[0]

    return [int(b) for b in bitstring[::-1]]

# --- Setup ---
N = 20 # Number of initial qubits to send
print(f"Generating {N} random bits and bases using Quantum Measurements...\n")

# === ALICE ===
alice_bits = get_quantum_random_bits(N)
alice_bases = get_quantum_random_bits(N) # 0 for Z basis (standard), 1 for X basis (diagonal)

# === BOB ===
bob_bases = get_quantum_random_bits(N)

# --- Quantum Transmission ---
qc = QuantumCircuit(N, N)

# === ALICE ===
for i in range(N):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

# === ALICE SENDS QUBITS TO BOB ===
qc.barrier()

# === BOB ===
for i in range(N):
    if bob_bases[i] == 1:
        qc.h(i)
    qc.measure(i, i)

# Run the simulation
backend = BasicSimulator()
t_qc = transpile(qc, backend)
result = backend.run(t_qc, shots=1).result()
counts = result.get_counts()
bob_bitstring = list(counts.keys())[0]
bob_bits = [int(b) for b in bob_bitstring[::-1]]

# --- Post-Processing (Classical Communication) ---
# === ALICE & BOB ===
sifted_alice_key = []
sifted_bob_key = []

# 1. Key Sifting phase
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_bits[i])

# 2. Error Checking (Sacrificing a subset)
# They publicly compare the first half of their sifted key [cite: 296, 298]
sample_size = len(sifted_alice_key) // 2
alice_sample = sifted_alice_key[:sample_size]
bob_sample = sifted_bob_key[:sample_size]

# The remaining bits become the final secure key
alice_final_key = sifted_alice_key[sample_size:]
bob_final_key = sifted_bob_key[sample_size:]

print("--- Protocol Results ---")
print(f"Alice's Bits:  {alice_bits}")
print(f"Alice's Bases: {alice_bases}")
print(f"Bob's Bases:   {bob_bases}")
print(f"Bob's Bits:    {bob_bits}")

print(f"\nSifted Key Length: {len(sifted_alice_key)}")
print(f"Sacrificed for testing: {sample_size} bits")

# Check for errors in the sample
errors = sum(1 for a, b in zip(alice_sample, bob_sample) if a != b)
error_rate = (errors / sample_size) if sample_size > 0 else 0

print(f"\nError Rate in Sample: {error_rate * 100:.2f}%")
if error_rate == 0:
    print("Protocol successful: No eavesdropping detected.")
    print(f"Alice's Final Key: {alice_final_key}")
    print(f"Bob's Final Key:   {bob_final_key}")
else:
    print("Error detected: Keys do not match. Discarding everything.")


Generating 20 random bits and bases using Quantum Measurements...

--- Protocol Results ---
Alice's Bits:  [0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1]
Alice's Bases: [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0]
Bob's Bases:   [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0]
Bob's Bits:    [1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1]

Sifted Key Length: 11
Sacrificed for testing: 5 bits

Error Rate in Sample: 0.00%
Protocol successful: No eavesdropping detected.
Alice's Final Key: [1, 0, 0, 1, 0, 1]
Bob's Final Key:   [1, 0, 0, 1, 0, 1]
